In [7]:
from sklearn.datasets import fetch_20newsgroups

categories = ["alt.atheism", "soc.religion.christian", "comp.graphics", "sci.med"]

twenty_train = fetch_20newsgroups(
    subset="train",
    categories=categories,
    shuffle=True,
    random_state=42
)

twenty_test = fetch_20newsgroups(
    subset="test",
    categories=categories,
    shuffle=True,
    random_state=42
)

print("Training samples:", len(twenty_train.data))
print("Testing samples:", len(twenty_test.data))
print("Categories:", twenty_train.target_names)

Training samples: 2257
Testing samples: 1502
Categories: ['alt.atheism', 'comp.graphics', 'sci.med', 'soc.religion.christian']


In [8]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.ensemble import RandomForestClassifier

text_clf = Pipeline([
    ("vect", CountVectorizer()),
    ("tfidf", TfidfTransformer()),
    ("clf", RandomForestClassifier(random_state=42))
])

text_clf.fit(twenty_train.data, twenty_train.target)

predicted = text_clf.predict(twenty_test.data)
accuracy = np.mean(predicted == twenty_test.target)

print("Bag-of-Words Accuracy:", accuracy)

Bag-of-Words Accuracy: 0.8035952063914781


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier

vectorizer = TfidfVectorizer(max_features=5000)

X_train = vectorizer.fit_transform(twenty_train.data)
X_test = vectorizer.transform(twenty_test.data)

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, twenty_train.target)

predicted = rf.predict(X_test)
accuracy = np.mean(predicted == twenty_test.target)

print("TF-IDF Word Feature Accuracy:", accuracy)

TF-IDF Word Feature Accuracy: 0.8155792276964048


In [10]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [11]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(twenty_train.data)

X_train_seq = tokenizer.texts_to_sequences(twenty_train.data)
X_test_seq = tokenizer.texts_to_sequences(twenty_test.data)

X_train_pad = pad_sequences(X_train_seq, padding="post", maxlen=500)
X_test_pad = pad_sequences(X_test_seq, padding="post", maxlen=500)

print("Training sequence shape:", X_train_pad.shape)
print("Testing sequence shape:", X_test_pad.shape)

Training sequence shape: (2257, 500)
Testing sequence shape: (1502, 500)


In [12]:
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers
from tensorflow.keras.losses import SparseCategoricalCrossentropy

model = Sequential()
model.add(layers.Embedding(input_dim=5000, output_dim=50, input_length=500))
model.add(layers.Flatten())
model.add(layers.Dense(10, activation="relu"))
model.add(layers.Dense(len(categories), activation="softmax"))

model.compile(
    optimizer="adam",
    loss=SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

model.summary()

/Users/vikasshukla/.pyenv/versions/3.11.6/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [13]:
history = model.fit(
    X_train_pad,
    twenty_train.target,
    epochs=5,
    batch_size=32,
    validation_data=(X_test_pad, twenty_test.target)
)

Epoch 1/5
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3379 - loss: 1.3511 - val_accuracy: 0.3968 - val_loss: 1.2735
Epoch 2/5
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5887 - loss: 0.9805 - val_accuracy: 0.7397 - val_loss: 0.6953
Epoch 3/5
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9518 - loss: 0.2153 - val_accuracy: 0.8129 - val_loss: 0.5039
Epoch 4/5
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9904 - loss: 0.0647 - val_accuracy: 0.8555 - val_loss: 0.4056
Epoch 5/5
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9987 - loss: 0.0210 - val_accuracy: 0.8562 - val_loss: 0.4170


In [14]:
loss, accuracy = model.evaluate(X_test_pad, twenty_test.target, verbose=0)

print("Custom Embedding Accuracy:", accuracy)

Custom Embedding Accuracy: 0.8561917543411255
